<a href="https://colab.research.google.com/github/ArmanHov2006/AI-Base-Recruitment-2/blob/main/piece_02_vecadd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu Aug  6 05:00:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from numba import cuda
print(cuda.is_available())
print(cuda.get_current_device().name)

True
Tesla T4


In [3]:
import numpy as np
from numba import cuda

N = 1000

a = np.arange(N, dtype=np.float32)
b = a * 2
ref = a + b

for i in range(5):
    print(f"a[{i}] = {a[i]:.1f}, b[{i}] = {b[i]:.1f}, ref[{i}] = {ref[i]:.1f}")

a[0] = 0.0, b[0] = 0.0, ref[0] = 0.0
a[1] = 1.0, b[1] = 2.0, ref[1] = 3.0
a[2] = 2.0, b[2] = 4.0, ref[2] = 6.0
a[3] = 3.0, b[3] = 6.0, ref[3] = 9.0
a[4] = 4.0, b[4] = 8.0, ref[4] = 12.0


In [7]:
@cuda.jit
def vecadd(a, b, out, n):
  i = cuda.grid(1)
  if i >= n:
    return
  out[i] = a[i] + b[i]


In [8]:
threads = 256
blocks = (N + threads - 1) // threads
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)

d_o = cuda.device_array(N, dtype=np.float32)
vecadd[blocks, threads](d_a, d_b, d_o, N)
out = d_o.copy_to_host()

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [10]:
ok = np.array_equal( out , ref )
print(f"all {N} matched: {ok}")

all 1000 matched: True
